In [1]:
!pip install deeplake transformers peft accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 57.3 MB/s eta 0:00:00


In [2]:
!pip install "deeplake<4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 643.4/643.4 kB 13.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 4.6 MB/s eta 0:00:00

In [3]:
import torch
import torch.nn as nn
import deeplake
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from datasets import Dataset

# ==========================================
# STEP 1: LOAD & FORMAT DATA
# ==========================================
print(">>> STEP 1: Streaming Data...")
ds = deeplake.load('hub://genai360/GAIR-lima-train-set', read_only=True)
ds_test = deeplake.load('hub://genai360/GAIR-lima-test-set', read_only=True)

/usr/local/lib/python3.12/dist-packages/deeplake/util/check_latest_version.py:32: UserWarning: A newer version of deeplake (4.4.2) is available. It's recommended that you update to the latest version using `pip install -U deeplake`.
  warnings.warn(


>>> STEP 1: Streaming Data...


-

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/genai360/GAIR-lima-train-set



/

hub://genai360/GAIR-lima-train-set loaded successfully.



|

This dataset can be visualized in Jupyter Notebook by ds.visualize() or at https://app.activeloop.ai/genai360/GAIR-lima-test-set



/

hub://genai360/GAIR-lima-test-set loaded successfully.



In [4]:
def prepare_sample_text(example):
    return f"Question: {example['question'].text()}\n\nAnswer: {example['answer'].text()}"

# Convert DeepLake to Standard Hugging Face Dataset
# (This avoids the TRL library errors we faced earlier)
def create_standard_dataset(deep_lake_ds):
    data_list = []
    for item in deep_lake_ds:
        data_list.append({"text": prepare_sample_text(item)})
    return Dataset.from_list(data_list)

hf_train_dataset = create_standard_dataset(ds)
hf_eval_dataset = create_standard_dataset(ds_test)
print(f"Data Loaded: {len(hf_train_dataset)} training samples.")


Data Loaded: 1030 training samples.


In [5]:
print(">>> STEP 2: Tokenizing...")
model_id = "facebook/opt-1.3b"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token # Fix padding for OPT

def tokenize_function(examples):
    # Truncate to 1024 to fit in memory
    return tokenizer(examples["text"], truncation=True, max_length=1024, padding="max_length")

# Apply tokenization to the standard dataset
tokenized_train = hf_train_dataset.map(tokenize_function, batched=True)
tokenized_eval = hf_eval_dataset.map(tokenize_function, batched=True)

>>> STEP 2: Tokenizing...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

Parameter 'function'=<function tokenize_function at 0x7c446e7bd260> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/1030 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [6]:
print(">>> STEP 3: Loading Model...")

# Check Hardware
device_type = torch.float16 if torch.cuda.is_available() else torch.float32
print(f"Using Precision: {device_type}")

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=device_type)

# Stability Hack: Freeze model and force Norms/Output to Float32
for param in model.parameters():
    param.requires_grad = False  # Freeze everything first
    if param.ndim == 1:
        param.data = param.data.to(torch.float32) # Cast LayerNorm to FP32

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

>>> STEP 3: Loading Model...
Using Precision: torch.float32


`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [7]:
class CastOutputToFloat(nn.Sequential):
    def forward(self, x): return super().forward(x).to(torch.float32)
model.lm_head = CastOutputToFloat(model.lm_head)

In [8]:
print(">>> STEP 4: Injecting LoRA Adapters...")
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# This is the magic line that adds the "Green Badges" (Trainable Params)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

>>> STEP 4: Injecting LoRA Adapters...
trainable params: 3,145,728 || all params: 1,318,903,808 || trainable%: 0.2385


In [9]:
print(">>> STEP 5: Initializing Trainer...")

training_args = TrainingArguments(
    output_dir="./OPT-fine_tuned-LIMA",
    per_device_train_batch_size=4, # Reduced to 4 to be safe on T4 GPU
    gradient_accumulation_steps=2, # Accumulate to simulate batch size 8
    learning_rate=1e-4,
    num_train_epochs=1,            # Reduced to 1 for quick testing
    logging_steps=10,
    fp16=True if torch.cuda.is_available() else False, # Use FP16 on GPU
    save_strategy="no",            # Don't save checkpoints to save disk space
    report_to="none",              # Disable WandB for simplicity
    remove_unused_columns=False    # Important for custom datasets
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

>>> STEP 5: Initializing Trainer...


In [12]:
import torch
from peft import PeftModel

# ==========================================
# STEP 1: MERGE (With Safety Check)
# ==========================================
print(">>> Attempting to merge model...")

# We check if the model has the 'merge_and_unload' method
# This prevents the "AttributeError" if you run the cell twice
if hasattr(model, "merge_and_unload"):
    model = model.merge_and_unload()
    print("✅ Merge successful! LoRA adapters fused into base model.")
else:
    print("ℹ️ Model is already merged (or is standard). Skipping merge step.")

# ==========================================
# STEP 2: SAVE TO DISK
# ==========================================
save_folder = "./OPT-LIMA-Merged-Final"
print(f">>> Saving model to {save_folder}...")

# We set safe_serialization=False to fix the OPT "Shared Tensors" error
model.save_pretrained(save_folder, safe_serialization=False)
tokenizer.save_pretrained(save_folder)

print("✅ Save complete!")

# ==========================================
# STEP 3: FINAL TEST (INFERENCE)
# ==========================================
print(">>> Running Test Inference...")

# 1. Prepare Input
prompt = "Question: How do I make a cup of tea?\n\nAnswer:"
inputs = tokenizer(prompt, return_tensors="pt")

# 2. Move to GPU/CPU
device = model.device
inputs = {k: v.to(device) for k, v in inputs.items()}

# 3. Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=64,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)

# 4. Print Result
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("-" * 30)
print(generated_text)
print("-" * 30)

>>> Attempting to merge model...
ℹ️ Model is already merged (or is standard). Skipping merge step.
>>> Saving model to ./OPT-LIMA-Merged-Final...
✅ Save complete!
>>> Running Test Inference...
------------------------------
Question: How do I make a cup of tea?

Answer: The best way to make a cup of tea is to steep the tea leaves in water. When the leaves are steeped, the water will cool and the leaves will become soft. Then you can strain the tea leaves through a fine sieve to remove the leaves that are not steeping. You can then add the tea
------------------------------


In [13]:
!pip install nbformat

import nbformat as nbf

path = "Fine-tuning using LoRA and SFT.ipynb"
nb = nbf.read(path, as_version=4)

# Remove broken widget metadata
if "widgets" in nb["metadata"]:
    del nb["metadata"]["widgets"]

nbf.write(nb, path)
print("Fixed notebook saved.")


FileNotFoundError: [Errno 2] No such file or directory: 'Fine-tuning using LoRA and SFT.ipynb'